# Hackathon Machine Learning -
## Previsione Abbandono Universitario

Siete stati contattati da una *grande università* che ha un problema urgente: troppi
studenti abbandonano gli studi prima della laurea, causando sia un danno
d'immagine che una perdita economica per l'ateneo. L'università vuole intervenire
proattivamente contattando gli studenti a rischio *prima che decidano di abbandonare*.
Hanno raccolto i dati di iscrizione e i risultati del primo anno accademico di circa
3.000 studenti. Il vostro compito è costruire un modello di Machine Learning capace
di **prevedere in anticipo chi si laureerà ("Graduate") e chi abbandonerà ("Dropout")**.

In [1]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
from sklearn.metrics import f1_score
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings('ignore')

print("Librerie caricate!")

Librerie caricate!


In [2]:
train = pd.read_csv('train_dropout.csv')
test  = pd.read_csv('test_dropout.csv')

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")
train.head()

Train: (2904, 37)
Test:  (726, 36)


,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,Admission grade,Displaced,Educational special needs,Debtor,Tuition fees up to date,Gender,Scholarship holder,Age at enrollment,International,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,44,1,9003,1,39,150.0,1,19,37,6,6,150.0,0,0,1,1,0,1,19,0,0,6,9,6,14.333333,0,0,6,7,6,13.833333,0,13.9,-0.3,0.79,Graduate
1,2,39,1,9991,0,1,135.0,1,34,34,0,0,134.1,1,0,0,0,1,0,49,0,0,5,5,0,0.000000,0,0,5,5,0,0.000000,0,7.6,2.6,0.32,Dropout
2,1,1,1,9238,1,1,121.0,1,1,1,1,1,115.1,1,0,0,1,0,0,19,0,0,6,10,6,10.833333,0,0,6,7,6,12.142857,0,11.1,0.6,2.02,Graduate
3,1,1,4,9500,1,1,135.0,1,37,37,9,8,123.0,1,0,0,1,0,1,18,0,0,8,8,7,13.381429,0,0,8,8,7,13.381429,0,10.8,1.4,1.74,Graduate
4,2,39,1,8014,0,12,110.0,1,34,34,0,0,157.8,0,0,0,1,0,0,44,0,5,8,13,8,13.750000,0,2,6,9,6,12.500000,0,7.6,2.6,0.32,Graduate


In [3]:
train = train.rename(columns={
    'Marital status': 'Stato civile', 
    'Application mode': 'Modalità di candidatura',
    'Application order': 'Ordine di candidatura',
    'Course': 'Corso',
    'Daytime/evening attendance\t': 'Frequenza diurna/serale',
    'Previous qualification': 'Qualifica precedente',
    'Previous qualification (grade)': 'Voto qualifica precedente',
    'Nacionality': 'Nazionalità',
    'Mother\'s qualification': 'Qualifica madre',
    'Father\'s qualification': 'Qualifica padre',
    'Mother\'s occupation': 'Occupazione madre',
    'Father\'s occupation': 'Occupazione padre',
    'Admission grade': 'Voto di ammissione',
    'Displaced': 'Sfollato',
    'Educational special needs': 'Bisogni educativi speciali',
    'Debtor': 'Debitore',
    'Tuition fees up to date': 'Tasse universitarie in regola',
    'Gender': 'Genere',
    'Scholarship holder': 'Titolare di borsa di studio',
    'Age at enrollment': 'Età all\'iscrizione',
    'International': 'Internazionale',
    'Curricular units 1st sem (credited)': 'Unità curriculari 1° sem (conseguite)',
    'Curricular units 1st sem (enrolled)': 'Unità curriculari 1° sem (iscritte)',
    'Curricular units 1st sem (evaluations)': 'Unità curriculari 1° sem (valutazioni)',
    'Curricular units 1st sem (approved)': 'Unità curriculari 1° sem (approvate)',
    'Curricular units 1st sem (grade)': 'Unità curriculari 1° sem (voto)',
    'Curricular units 1st sem (without evaluations)': 'Unità curriculari 1° sem (senza valutazioni)',
    'Curricular units 2nd sem (credited)': 'Unità curriculari 2° sem (con crediti)',
    'Curricular units 2nd sem (enrolled)': 'Unità curriculari 2° sem (iscritte)',
    'Curricular units 2nd sem (evaluations)': 'Unità curriculari 2° sem (valutazioni)',
    'Curricular units 2nd sem (approved)': 'Unità curriculari 2° sem (approvate)',
    'Curricular units 2nd sem (grade)': 'Unità curriculari 2° sem (voto)',
    'Curricular units 2nd sem (without evaluations)': 'Unità curriculari 2° sem (senza valutazioni)',
    'Unemployment rate': 'Tasso di disoccupazione',
    'Inflation rate': 'Tasso di inflazione',
    'GDP': 'PIL',

})

test.head()

test = test.rename(columns={
    'Marital status': 'Stato civile', 
    'Application mode': 'Modalità di candidatura',
    'Application order': 'Ordine di candidatura',
    'Course': 'Corso',
    'Daytime/evening attendance\t': 'Frequenza diurna/serale',
    'Previous qualification': 'Qualifica precedente',
    'Previous qualification (grade)': 'Voto qualifica precedente',
    'Nacionality': 'Nazionalità',
    'Mother\'s qualification': 'Qualifica madre',
    'Father\'s qualification': 'Qualifica padre',
    'Mother\'s occupation': 'Occupazione madre',
    'Father\'s occupation': 'Occupazione padre',
    'Admission grade': 'Voto di ammissione',
    'Displaced': 'Sfollato',
    'Educational special needs': 'Bisogni educativi speciali',
    'Debtor': 'Debitore',
    'Tuition fees up to date': 'Tasse universitarie in regola',
    'Gender': 'Genere',
    'Scholarship holder': 'Titolare di borsa di studio',
    'Age at enrollment': 'Età all\'iscrizione',
    'International': 'Internazionale',
    'Curricular units 1st sem (credited)': 'Unità curriculari 1° sem (conseguite)',
    'Curricular units 1st sem (enrolled)': 'Unità curriculari 1° sem (iscritte)',
    'Curricular units 1st sem (evaluations)': 'Unità curriculari 1° sem (valutazioni)',
    'Curricular units 1st sem (approved)': 'Unità curriculari 1° sem (approvate)',
    'Curricular units 1st sem (grade)': 'Unità curriculari 1° sem (voto)',
    'Curricular units 1st sem (without evaluations)': 'Unità curriculari 1° sem (senza valutazioni)',
    'Curricular units 2nd sem (credited)': 'Unità curriculari 2° sem (con crediti)',
    'Curricular units 2nd sem (enrolled)': 'Unità curriculari 2° sem (iscritte)',
    'Curricular units 2nd sem (evaluations)': 'Unità curriculari 2° sem (valutazioni)',
    'Curricular units 2nd sem (approved)': 'Unità curriculari 2° sem (approvate)',
    'Curricular units 2nd sem (grade)': 'Unità curriculari 2° sem (voto)',
    'Curricular units 2nd sem (without evaluations)': 'Unità curriculari 2° sem (senza valutazioni)',
    'Unemployment rate': 'Tasso di disoccupazione',
    'Inflation rate': 'Tasso di inflazione',
    'GDP': 'PIL',

})

test.head()

,Stato civile,Modalità di candidatura,Ordine di candidatura,Corso,Frequenza diurna/serale,Qualifica precedente,Voto qualifica precedente,Nazionalità,Qualifica madre,Qualifica padre,Occupazione madre,Occupazione padre,Voto di ammissione,Sfollato,Bisogni educativi speciali,Debitore,Tasse universitarie in regola,Genere,Titolare di borsa di studio,Età all'iscrizione,Internazionale,Unità curriculari 1° sem (conseguite),Unità curriculari 1° sem (iscritte),Unità curriculari 1° sem (valutazioni),Unità curriculari 1° sem (approvate),Unità curriculari 1° sem (voto),Unità curriculari 1° sem (senza valutazioni),Unità curriculari 2° sem (con crediti),Unità curriculari 2° sem (iscritte),Unità curriculari 2° sem (valutazioni),Unità curriculari 2° sem (approvate),Unità curriculari 2° sem (voto),Unità curriculari 2° sem (senza valutazioni),Tasso di disoccupazione,Tasso di inflazione,PIL
0,1,43,1,9147,1,1,129.0,1,19,1,9,5,112.0,0,0,0,1,1,0,22,0,5,9,9,5,11.200000,0,5,9,14,7,11.250000,2,12.7,3.7,-1.70
1,1,53,1,9254,1,42,110.0,1,1,1,4,7,111.8,1,0,0,1,0,1,21,0,0,6,7,6,10.571429,0,0,6,8,5,11.000000,0,8.9,1.4,3.51
2,1,43,1,9500,1,1,140.0,1,19,19,7,7,138.6,0,0,0,1,0,1,19,0,0,7,9,6,14.250000,0,0,8,11,8,13.063636,0,16.2,0.3,-0.92
3,1,1,5,9500,1,1,118.0,1,3,1,5,10,113.5,1,0,0,1,0,0,20,0,0,7,7,7,14.307143,0,0,8,8,8,15.012500,0,12.4,0.5,1.79
4,1,1,2,9556,1,1,142.0,1,1,19,4,5,130.5,1,0,0,1,0,1,18,0,0,7,8,7,12.214286,0,0,8,8,8,14.062500,0,16.2,0.3,-0.92


*Quanti laureati/ritirati ci sono?*

In [4]:
print(train['Target'].value_counts())
print(train['Target'].value_counts(normalize=True) * 100) # percentuale

Target
Graduate    1767
Dropout     1137
Name: count, dtype: int64
Target
Graduate    60.847107
Dropout     39.152893
Name: proportion, dtype: float64


In [5]:
print(train.isnull().sum().sum())
print(test.isnull().sum().sum())

0
0


**0 Valori Nulli**

In [6]:
train.describe().round(2) # .round(2) per limitare a 2 decimali

,Stato civile,Modalità di candidatura,Ordine di candidatura,Corso,Frequenza diurna/serale,Qualifica precedente,Voto qualifica precedente,Nazionalità,Qualifica madre,Qualifica padre,Occupazione madre,Occupazione padre,Voto di ammissione,Sfollato,Bisogni educativi speciali,Debitore,Tasse universitarie in regola,Genere,Titolare di borsa di studio,Età all'iscrizione,Internazionale,Unità curriculari 1° sem (conseguite),Unità curriculari 1° sem (iscritte),Unità curriculari 1° sem (valutazioni),Unità curriculari 1° sem (approvate),Unità curriculari 1° sem (voto),Unità curriculari 1° sem (senza valutazioni),Unità curriculari 2° sem (con crediti),Unità curriculari 2° sem (iscritte),Unità curriculari 2° sem (valutazioni),Unità curriculari 2° sem (approvate),Unità curriculari 2° sem (voto),Unità curriculari 2° sem (senza valutazioni),Tasso di disoccupazione,Tasso di inflazione,PIL
count,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00,2904.00
mean,1.19,18.42,1.75,8848.06,0.89,4.62,132.99,1.76,20.03,22.59,10.15,10.31,127.25,0.54,0.01,0.11,0.87,0.34,0.27,23.48,0.02,0.76,6.34,8.06,4.79,10.52,0.13,0.59,6.29,7.75,4.51,10.01,0.13,11.66,1.23,0.00
std,0.61,17.34,1.33,2077.05,0.31,10.16,13.18,6.21,15.59,15.26,23.50,22.52,14.82,0.50,0.10,0.32,0.34,0.47,0.44,7.88,0.15,2.48,2.57,4.30,3.23,5.06,0.67,2.01,2.25,3.98,3.16,5.50,0.72,2.67,1.38,2.25
min,1.00,1.00,0.00,33.00,0.00,1.00,95.00,1.00,1.00,1.00,0.00,0.00,95.00,0.00,0.00,0.00,0.00,0.00,0.00,17.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,7.60,-0.80,-4.06
25%,1.00,1.00,1.00,9085.00,1.00,1.00,125.00,1.00,2.00,3.00,4.00,4.00,118.00,0.00,0.00,0.00,1.00,0.00,0.00,19.00,0.00,0.00,5.00,6.00,3.00,11.00,0.00,0.00,5.00,6.00,2.00,10.50,0.00,9.40,0.30,-1.70
50%,1.00,17.00,1.00,9238.00,1.00,1.00,133.10,1.00,19.00,19.00,5.00,7.00,126.30,1.00,0.00,0.00,1.00,0.00,0.00,20.00,0.00,0.00,6.00,8.00,5.00,12.38,0.00,0.00,6.00,8.00,5.00,12.33,0.00,11.10,1.40,0.32
75%,1.00,39.00,2.00,9556.00,1.00,1.00,140.00,1.00,37.00,37.00,9.00,9.00,135.43,1.00,0.00,0.00,1.00,1.00,1.00,26.00,0.00,0.00,7.00,10.00,6.00,13.50,0.00,0.00,7.00,10.00,6.00,13.50,0.00,13.90,2.60,1.79
max,6.00,57.00,6.00,9991.00,1.00,43.00,190.00,105.00,44.00,44.00,194.00,194.00,190.00,1.00,1.00,1.00,1.00,1.00,1.00,70.00,1.00,20.00,26.00,45.00,26.00,18.00,12.00,19.00,23.00,33.00,20.00,18.57,12.00,16.20,3.70,3.51


### Osserviamo le variabili più correlate con il target

In [7]:
# Creazione di una colonna numerica per il Target (1 se Dropout, 0 se No Dropout)
train_temp = train.copy()
train_temp['Target_num'] = (train_temp['Target'] == 'Dropout').astype(int)

# Correlazione di ogni colonna con il Target
correlazioni = (train_temp
                .drop(columns=['Target'])
                .corr()['Target_num']
                .drop('Target_num')
                .abs()
                .sort_values(ascending=False)) 

"""
 Cosa abbiamo fatto: abbiamo calcolato la correlazione di ogni colonna con il Target,
 abbiamo preso il valore assoluto, abbiamo ordinato in modo decrescente e abbiamo tolto
 la colonna Target_num che è quella che abbiamo creato per fare i calcoli
 """

print("Top 15 variabili più correlate con Dropout:")
print(correlazioni.head(15).round(3)) # .round(3) per limitare a 3 decimali

Top 15 variabili più correlate con Dropout:
Unità curriculari 2° sem (approvate)      0.652
Unità curriculari 2° sem (voto)           0.609
Unità curriculari 1° sem (approvate)      0.552
Unità curriculari 1° sem (voto)           0.524
Tasse universitarie in regola             0.438
Titolare di borsa di studio               0.302
Debitore                                  0.269
Età all'iscrizione                        0.264
Modalità di candidatura                   0.255
Genere                                    0.249
Unità curriculari 2° sem (iscritte)       0.178
Unità curriculari 1° sem (iscritte)       0.156
Voto di ammissione                        0.132
Sfollato                                  0.120
Unità curriculari 2° sem (valutazioni)    0.119
Name: Target_num, dtype: float64


*Variabili con più correlazione -> inerenti agli esami passati e ai voti del **secondo semestre**, poi quelli del primo*

In [8]:
y = (train['Target'] == 'Dropout').astype(int)
X = train.drop(columns=['Target'])
print(f"X: {X.shape}, y: {y.value_counts().to_dict()}")

X: (2904, 36), y: {0: 1767, 1: 1137}


### Feature Engeneering su train -> creiamo 4 feature che derivano da un ragionamento logico sulle correlazioni trovate prima

In [9]:
# Trend esami: calo tra 1° e 2° semestre
X['trend_approvati'] = X['Unità curriculari 2° sem (approvate)'] - X['Unità curriculari 1° sem (approvate)']

# Totale esami approvati nell'anno
X['totale_approvati'] = X['Unità curriculari 2° sem (approvate)'] + X['Unità curriculari 1° sem (approvate)']

# Voto medio tra i due semestri
X['voto_medio'] = (X['Unità curriculari 1° sem (voto)'] + X['Unità curriculari 2° sem (voto)']) / 2

# Rischio finanziario
X['rischio_finanziario'] = X['Debitore'] + (1 - X['Tasse universitarie in regola']) - X['Titolare di borsa di studio']

print(f"Colonne dopo feature engineering: {X.shape[1]}")

Colonne dopo feature engineering: 40


In [10]:
X_test_final = test.copy()

X_test_final['trend_approvati'] = X_test_final['Unità curriculari 2° sem (approvate)'] - X_test_final['Unità curriculari 1° sem (approvate)']

# Totale esami approvati nell'anno
X_test_final['totale_approvati'] = X_test_final['Unità curriculari 2° sem (approvate)'] + X_test_final['Unità curriculari 1° sem (approvate)']

# Voto medio tra i due semestri
X_test_final['voto_medio'] = (X_test_final['Unità curriculari 1° sem (voto)'] + X_test_final['Unità curriculari 2° sem (voto)']) / 2

# Rischio finanziario
X_test_final['rischio_finanziario'] = X_test_final['Debitore'] + (1 - X_test_final['Tasse universitarie in regola']) - X_test_final['Titolare di borsa di studio']

print(f"X_test_final pronto: {X_test_final.shape}")

X_test_final pronto: (726, 40)


### Split in training e validation del training dataset

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training:   {X_train.shape[0]} righe")
print(f"Validation: {X_val.shape[0]} righe")
print(f"Dropout nel training:   {y_train.mean():.2%}")
print(f"Dropout nel validation: {y_val.mean():.2%}")

Training:   2323 righe
Validation: 581 righe
Dropout nel training:   39.17%
Dropout nel validation: 39.07%


### Creazione e allenamento modelli con Pipeline

In [12]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.svm import SVC
from catboost import CatBoostClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scala = (y_train == 0).sum() / (y_train == 1).sum() # calcolo del rapporto tra classi per bilanciare XGBoost

# ── Logistic Regression ──────────────────────────────────────
pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('modello', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])
param_lr = {
    'modello__C': [0.01, 0.1, 1, 10], # forza della regolarizzazione
}

# ── Random Forest ─────────────────────────────────────────────
pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('modello', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1))
])
param_rf = {
    'modello__n_estimators': [200, 300, 500], # numero di alberi nella foresta
    'modello__max_depth':    [None, 10, 20], # profondità massima degli alberi (None = senza limite)
}

# ── XGBoost ───────────────────────────────────────────────────
pipeline_xgb = Pipeline([
    ('scaler', StandardScaler()),
    ('modello', XGBClassifier(scale_pos_weight=scala, random_state=42, eval_metric='logloss', n_jobs=-1))
])
param_xgb = {
    'modello__n_estimators':  [300, 500], # numero di alberi nella foresta
    'modello__learning_rate': [0.01, 0.05, 0.1], # passo di apprendimento
    'modello__max_depth':     [4, 6, 8], # profondità massima degli alberi
}

# ── LightGBM ──────────────────────────────────────────────────
pipeline_lgbm = Pipeline([
    ('scaler', StandardScaler()),
    ('modello', lgb.LGBMClassifier(class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1))
])
param_lgbm = {
    'modello__n_estimators':  [300, 500], # numero di alberi nella foresta
    'modello__learning_rate': [0.01, 0.05, 0.1], # passo di apprendimento
    'modello__max_depth':     [4, 6, 8], # profondità massima degli alberi
}

# ── Extra Trees ───────────────────────────────────────────────
pipeline_et = Pipeline([
    ('scaler', StandardScaler()),
    ('modello', ExtraTreesClassifier(class_weight='balanced', random_state=42, n_jobs=-1))
])
param_et = {
    'modello__n_estimators': [200, 300, 500],
    'modello__max_depth':    [None, 10, 20],
}

# ── SVM ───────────────────────────────────────────────────────
pipeline_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('modello', SVC(class_weight='balanced', random_state=42))
])
param_svm = {
    'modello__C':      [0.1, 1, 10],
    'modello__kernel': ['rbf', 'linear'],
}

# ── CatBoost ──────────────────────────────────────────────────
pipeline_cat = Pipeline([
    ('scaler', StandardScaler()),
    ('modello', CatBoostClassifier(
        auto_class_weights='Balanced',
        random_state=42,
        verbose=0
    ))
])
param_cat = {
    'modello__iterations':    [300, 500],
    'modello__learning_rate': [0.01, 0.05, 0.1],
    'modello__depth':         [4, 6, 8],
}
print("Pipeline e griglie definite!")

Pipeline e griglie definite!


### Tuning iperparametri per ogni modello

In [13]:
configurazioni = {
    'Logistic Regression': (pipeline_lr, param_lr),
    'Random Forest':       (pipeline_rf, param_rf),
    'XGBoost':             (pipeline_xgb, param_xgb),
    'LightGBM':            (pipeline_lgbm, param_lgbm),
    'Extra Trees':         (pipeline_et, param_et),
    'SVM':                 (pipeline_svm, param_svm),
    'CatBoost':            (pipeline_cat, param_cat),
}

risultati_tuning = {}
migliori_pipeline = {}

for nome, (pipeline, params) in configurazioni.items():
    
    gs = GridSearchCV(
        pipeline,
        params,
        scoring='f1_macro',
        cv=cv,
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    
    y_pred = gs.best_estimator_.predict(X_val)
    f1 = f1_score(y_val, y_pred, average='macro')
    
    risultati_tuning[nome] = f1
    migliori_pipeline[nome] = gs.best_estimator_
    
    print(f"{nome:<25} F1 Macro: {f1:.4f} | Params: {gs.best_params_}")

Logistic Regression       F1 Macro: 0.8857 | Params: {'modello__C': 0.1}
Random Forest             F1 Macro: 0.8910 | Params: {'modello__max_depth': 20, 'modello__n_estimators': 200}
XGBoost                   F1 Macro: 0.8868 | Params: {'modello__learning_rate': 0.05, 'modello__max_depth': 4, 'modello__n_estimators': 300}
LightGBM                  F1 Macro: 0.8995 | Params: {'modello__learning_rate': 0.01, 'modello__max_depth': 8, 'modello__n_estimators': 500}
Extra Trees               F1 Macro: 0.8943 | Params: {'modello__max_depth': 20, 'modello__n_estimators': 500}
SVM                       F1 Macro: 0.8805 | Params: {'modello__C': 0.1, 'modello__kernel': 'linear'}
CatBoost                  F1 Macro: 0.9030 | Params: {'modello__depth': 6, 'modello__iterations': 500, 'modello__learning_rate': 0.05}


**Proviamo ad ampliare le griglie di parametri per migliorare F1 score dei modelli**

In [14]:
# Griglie ampliate per i modelli ai bordi
param_lgbm_v2 = {
    'modello__n_estimators':  [500, 700, 1000],
    'modello__learning_rate': [0.01, 0.05],
    'modello__max_depth':     [8, 10, 12],
}

param_et_v2 = {
    'modello__n_estimators': [500, 700, 1000],
    'modello__max_depth':    [20, 30, None],
}

param_cat_v2 = {
    'modello__iterations':    [500, 700, 1000],
    'modello__learning_rate': [0.01, 0.05],
    'modello__depth':         [6, 8, 10],
}

param_rf_v2 = {
    'modello__n_estimators': [300, 500],
    'modello__max_depth':    [20, 30, None],
}

configurazioni_v2 = {
    'LightGBM':    (pipeline_lgbm, param_lgbm_v2),
    'Extra Trees': (pipeline_et,   param_et_v2),
    'CatBoost':    (pipeline_cat,  param_cat_v2),
    'Random Forest': (pipeline_rf, param_rf_v2),
}

for nome, (pipeline, params) in configurazioni_v2.items():
    
    gs = GridSearchCV(
        pipeline,
        params,
        scoring='f1_macro',
        cv=cv,
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    
    y_pred = gs.best_estimator_.predict(X_val)
    f1 = f1_score(y_val, y_pred, average='macro')
    
    risultati_tuning[nome] = f1
    migliori_pipeline[nome] = gs.best_estimator_
    
    print(f"✅ {nome:<25} F1 Macro: {f1:.4f} | Params: {gs.best_params_}")

✅ LightGBM                  F1 Macro: 0.9030 | Params: {'modello__learning_rate': 0.01, 'modello__max_depth': 8, 'modello__n_estimators': 1000}
✅ Extra Trees               F1 Macro: 0.9015 | Params: {'modello__max_depth': 20, 'modello__n_estimators': 1000}
✅ CatBoost                  F1 Macro: 0.9012 | Params: {'modello__depth': 6, 'modello__iterations': 1000, 'modello__learning_rate': 0.05}
✅ Random Forest             F1 Macro: 0.8929 | Params: {'modello__max_depth': 20, 'modello__n_estimators': 300}


### ciao